# 🎙️ MASTER OF MASTERS — MOTOR NEURAL HD DE CLONAGEM DE VOZ (48kHz / 24-BIT)
### Pipeline de Alta Fidelidade: RVC v2 (48kHz) + RMVPE + FAISS Index + BS-Roformer

Este notebook roda **100% gratuito** na GPU Nvidia T4 da Google e executa a conversão neural real para o **Master of Masters Studio Pro**.

---
### 📋 Instruções:
1. No menu superior do Google Colab, clique em **Ambiente de Execução > Alterar tipo de ambiente de execução** e escolha **T4 GPU** (Gratuito).
2. Execute a **Célula 1** (Instalação dos Pesos e Dependências RVC v2 48kHz).
3. Se você for treinar o seu timbre pela primeira vez, faça upload do seu áudio na pasta `/content/minha_voz.wav` e execute a **Célula 2**.
4. Execute a **Célula 3** (Iniciar API Pública para conectar direto ao seu Studio).

In [ ]:
#@title ⚡ CÉLULA 1: Instalação do Motor RVC v2 (48kHz + RMVPE + CUDA)
!nvidia-smi

import os, sys
print("📥 Baixando e configurando RVC v2 48kHz...")
!apt-get -y update && apt-get -y install ffmpeg sox

# Clonar motor RVC v2 otimizado
!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/RVC
%cd /content/RVC
!pip install -r requirements.txt

# Baixar modelos fundamentais ContentVec 768, RMVPE e Pretrained v2 48k
!mkdir -p /content/RVC/assets/hubert
!mkdir -p /content/RVC/assets/rmvpe
!mkdir -p /content/RVC/assets/pretrained_v2

!wget -nc -O /content/RVC/assets/hubert/hubert_base.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt
!wget -nc -O /content/RVC/assets/rmvpe/rmvpe.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt
!wget -nc -O /content/RVC/assets/pretrained_v2/f0G48k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G48k.pth
!wget -nc -O /content/RVC/assets/pretrained_v2/f0D48k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D48k.pth

print("\n✅ RVC v2 48kHz e RMVPE instalados e prontos para GPU CUDA!")

In [ ]:
#@title 🎙️ CÉLULA 2: Treinar Modelo da Sua Voz Real (48kHz HD)
# Faça upload de 'minha_voz.wav' (3 a 10 min cantando/falando) na pasta /content/

MODEL_NAME = "MinhaVozReal" #@param {type:"string"}
DATASET_AUDIO = "/content/minha_voz.wav" #@param {type:"string"}
EPOCHS = 100 #@param {type:"integer"}

import os
%cd /content/RVC
dataset_dir = f"/content/dataset_{MODEL_NAME}"
os.makedirs(dataset_dir, exist_ok=True)

if os.path.exists(DATASET_AUDIO):
    print(f"📦 Fatiando e normalizando dataset em 48kHz...")
    !ffmpeg -i "{DATASET_AUDIO}" -ar 48000 -ac 1 -f segment -segment_time 4 "{dataset_dir}/chunk_%03d.wav"
    
    print(f"🔍 Extraindo pitch com RMVPE e criando índice de características FAISS...")
    # Extrair pitch e features
    !python infer/modules/train/preprocess.py "{dataset_dir}" 48000 2 "/content/RVC/logs/{MODEL_NAME}" False 3.0
    !python infer/modules/train/extract/extract_f0_rmvpe.py 1 0 0 "/content/RVC/logs/{MODEL_NAME}" True
    !python infer/modules/train/extract_feature_print.py cuda:0 1 0 0 "/content/RVC/logs/{MODEL_NAME}" v2
    
    print(f"🚀 Treinando rede neural RVC v2 48k por {EPOCHS} épocas...")
    !python infer/modules/train/train.py -e "{MODEL_NAME}" -sr 48k -f0 1 -bs 8 -g 0 -te {EPOCHS} -se 10 -pg assets/pretrained_v2/f0G48k.pth -pd assets/pretrained_v2/f0D48k.pth -l 0 -c 0 -sw 0 -v v2
    
    # Treinar e salvar índice FAISS
    !python infer/modules/train/train_index.py "/content/RVC/logs/{MODEL_NAME}" v2
    print(f"\n🎉 TREINO CONCLUÍDO COM SUCESSO! Modelo '{MODEL_NAME}.pth' pronto para uso!")
else:
    print(f"⚠️ Arquivo '{DATASET_AUDIO}' não encontrado. Faça upload do seu áudio na pasta /content/ com o nome 'minha_voz.wav'.")

In [ ]:
#@title 🌐 CÉLULA 3: Iniciar Servidor de Inferência Neural HD (API para o Studio)
import os, glob, torch, soundfile as sf, numpy as np, io
import gradio as gr

%cd /content/RVC
from configs.config import Config
from infer.modules.vc.modules import VC

config = Config()
config.device = "cuda:0" if torch.cuda.is_available() else "cpu"
config.is_half = True if torch.cuda.is_available() else False
vc = VC(config)

def get_available_models():
    models = [os.path.basename(p) for p in glob.glob("/content/RVC/assets/weights/*.pth")]
    return models if models else ["MinhaVozReal.pth"]

print("🚀 Inicializando pipeline de inferência RVC v2 48kHz acelerado por GPU...")

def convert_vocal_hd(audio_file, model_name="MinhaVozReal", pitch_shift=0, index_rate=0.85, protect_rate=0.33):
    """
    Executa a conversão neural real usando o modelo RVC v2, extrator RMVPE e índice FAISS.
    """
    if audio_file is None:
        return None, "Nenhum áudio recebido."
    
    sr, data = audio_file
    input_path = "/content/temp_input.wav"
    output_path = "/content/temp_output.wav"
    sf.write(input_path, data, sr)
    
    # Carregar modelo .pth
    clean_model = model_name if model_name.endswith(".pth") else f"{model_name}.pth"
    weight_path = f"/content/RVC/assets/weights/{clean_model}"
    if not os.path.exists(weight_path):
        weights = glob.glob("/content/RVC/assets/weights/*.pth")
        if weights:
            weight_path = weights[0]
            clean_model = os.path.basename(weight_path)
        else:
            return audio_file, f"⚠️ Nenhum modelo .pth encontrado em assets/weights. Treine na Célula 2 primeiro."
            
    vc.get_vc(clean_model)
    
    # Localizar arquivo de índice FAISS correspondente
    model_base = clean_model.replace(".pth", "")
    index_files = glob.glob(f"/content/RVC/logs/{model_base}/added_*.index")
    index_path = index_files[0] if index_files else ""
    
    # Executar inferência neural de alta definição
    info, opt = vc.vc_single(
        sid=0,
        input_audio_path=input_path,
        f0_up_key=int(pitch_shift),
        f0_file=None,
        f0_method="rmvpe",
        file_index=index_path,
        file_index2="",
        index_rate=float(index_rate),
        filter_radius=3,
        resample_sr=48000,
        rms_mix_rate=0.25,
        protect=float(protect_rate)
    )
    
    if opt is not None:
        out_sr, out_data = opt
        return (out_sr, out_data), f"✅ Voz convertida com sucesso ({clean_model} @ 48kHz RMVPE)!"
    
    return audio_file, f"Status: {info}"

demo = gr.Interface(
    fn=convert_vocal_hd,
    inputs=[
        gr.Audio(label="Vocal da Música (Suno/Master)", type="numpy"),
        gr.Textbox(value="MinhaVozReal", label="Modelo de Voz (.pth)"),
        gr.Slider(-12, 12, value=0, step=1, label="Transpose (Semitons)"),
        gr.Slider(0.0, 1.0, value=0.85, step=0.05, label="FAISS Index Rate (Fidelidade do Timbre)"),
        gr.Slider(0.0, 0.5, value=0.33, step=0.01, label="Proteção de Consoantes/Respiração")
    ],
    outputs=[
        gr.Audio(label="Vocal com Sua Voz Real (HD 48kHz)"),
        gr.Textbox(label="Status do Processamento")
    ],
    title="🎙️ Master of Masters — HD Neural Voice Engine (48kHz RVC v2)"
)

# Lançar túnel público Gradio para comunicação com o Studio
demo.launch(share=True, debug=False)